# Caloric Intake Regression Pipeline

This notebook builds and evaluates regression models to predict `calories_goal_per_day` from `caloric_intake_rice_age_dataset_385.csv`.

In [ ]:
import io
import random
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set(style='whitegrid')

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Guarded xgboost import/install
try:
    from xgboost import XGBRegressor
except ImportError:
    try:
        install_choice = input('xgboost is not installed. Install it now with pip? [Y/n]: ').strip().lower()
    except EOFError:
        install_choice = 'y'

    if install_choice in ('y', ''):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost'])
        from xgboost import XGBRegressor
    else:
        raise ImportError('xgboost is required for this notebook. Please install it and rerun this cell.')

print(f'Global random seed set to {SEED}')


In [ ]:
# Load dataset from repository root
DATA_PATH = 'caloric_intake_rice_age_dataset_385.csv'
df = pd.read_csv(DATA_PATH)

print('Shape:', df.shape)
print()
print('Head:')
print(df.head())

print()
print('Info:')
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

print('Missing values per column:')
print(df.isna().sum())

In [ ]:
# Target distribution summary
TARGET_COL = 'calories_goal_per_day'

print('Target describe:')
print(df[TARGET_COL].describe())

plt.figure(figsize=(8, 5))
sns.histplot(df[TARGET_COL], kde=True, bins=30)
plt.title('Target Distribution: calories_goal_per_day')
plt.xlabel(TARGET_COL)
plt.ylabel('Count')
plt.show()

In [ ]:
# Feature/target split and train/test split
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print('Categorical columns:', categorical_cols)
print('Numerical columns:', numerical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

In [ ]:
# Preprocessing pipelines
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols),
    ]
)

In [ ]:
# Cross-validation strategy
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

In [ ]:
# Build and tune models
models = {}

# Linear Regression baseline
linear_pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LinearRegression()),
    ]
)
linear_pipe.fit(X_train, y_train)
models['LinearRegression'] = linear_pipe

# Ridge with simple alpha tuning
ridge_pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', Ridge()),
    ]
)
ridge_grid = {
    'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
}
ridge_search = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
ridge_search.fit(X_train, y_train)
models['Ridge'] = ridge_search.best_estimator_

print('Ridge best params:', ridge_search.best_params_)
print('Ridge best CV score (neg RMSE):', ridge_search.best_score_)

In [ ]:
# RandomForest hyperparameter tuning with GridSearchCV
rf_pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(random_state=SEED)),
    ]
)

rf_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None, 8, 16],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
}

rf_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_grid,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
models['RandomForestRegressor'] = rf_search.best_estimator_

print('RandomForest best params:', rf_search.best_params_)
print('RandomForest best CV score (neg RMSE):', rf_search.best_score_)

In [ ]:
# XGBRegressor hyperparameter tuning with RandomizedSearchCV
xgb_pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', XGBRegressor(
            random_state=SEED,
            objective='reg:squarederror',
            n_jobs=-1,
            eval_metric='rmse',
        )),
    ]
)

xgb_param_dist = {
    'model__n_estimators': [200, 400, 600],
    'model__max_depth': [3, 4, 5, 6],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.85, 1.0],
    'model__colsample_bytree': [0.7, 0.85, 1.0],
    'model__reg_alpha': [0.0, 0.1, 1.0],
    'model__reg_lambda': [1.0, 5.0, 10.0],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=xgb_param_dist,
    n_iter=30,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=SEED,
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
models['XGBRegressor'] = xgb_search.best_estimator_

print('XGBRegressor best params:', xgb_search.best_params_)
print('XGBRegressor best CV score (neg RMSE):', xgb_search.best_score_)

In [ ]:
# Test-set evaluation + diagnostic plots
results = []
predictions = {}

for model_name, fitted_model in models.items():
    y_pred = fitted_model.predict(X_test)
    predictions[model_name] = y_pred

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
    })

    print(f'\n{model_name} test metrics')
    print(f'MAE:  {mae:.4f}')
    print(f'RMSE: {rmse:.4f}')
    print(f'R2:   {r2:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Predicted vs actual
    axes[0].scatter(y_test, y_pred, alpha=0.75)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    axes[0].set_title(f'{model_name}: Predicted vs Actual')
    axes[0].set_xlabel('Actual')
    axes[0].set_ylabel('Predicted')

    # Residual plot
    residuals = y_test - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.75)
    axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_title(f'{model_name}: Residuals vs Predicted')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Residuals')

    plt.tight_layout()
    plt.show()

In [ ]:
# RandomForest feature importance (top 15)
rf_best = models['RandomForestRegressor']
rf_preprocessor = rf_best.named_steps['preprocessor']
rf_model = rf_best.named_steps['model']

rf_feature_names = rf_preprocessor.get_feature_names_out()
rf_importances = rf_model.feature_importances_

rf_importance_df = pd.DataFrame({
    'feature': rf_feature_names,
    'importance': rf_importances,
}).sort_values('importance', ascending=False)

print('Top 15 RandomForest features:')
print(rf_importance_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(
    data=rf_importance_df.head(15),
    x='importance',
    y='feature',
)
plt.title('RandomForest Top 15 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# XGBRegressor feature importance (top 15)
xgb_best = models['XGBRegressor']
xgb_preprocessor = xgb_best.named_steps['preprocessor']
xgb_model = xgb_best.named_steps['model']

xgb_feature_names = xgb_preprocessor.get_feature_names_out()

if hasattr(xgb_model, 'feature_importances_'):
    xgb_importance_df = pd.DataFrame({
        'feature': xgb_feature_names,
        'importance': xgb_model.feature_importances_,
    }).sort_values('importance', ascending=False)

    print('Top 15 XGBRegressor features:')
    print(xgb_importance_df.head(15))

    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=xgb_importance_df.head(15),
        x='importance',
        y='feature',
    )
    plt.title('XGBRegressor Top 15 Feature Importances')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
else:
    print('XGBRegressor feature importances are not available for this configuration.')

In [ ]:
# Summary
summary_df = pd.DataFrame(results).sort_values(by='RMSE', ascending=True).reset_index(drop=True)
print('Model comparison (sorted by RMSE):')
print(summary_df.to_string(index=False))
summary_df